# Composable NLP pipeline

In this notebook you will compose plugins from the `textpipeline` library. Each call to `process` handles one review and returns that review's final Tokens. You can only import from the `textpipelien` library or Python built-in libraries.

In [ ]:
import re

from textpipeline import (
    CasingNormalizer,
    Lemmatizer,
    NGramGenerator,
    Preprocessor,
    SimpleStemmer,
    SpecialCharacterRemover,
    StopwordRemover,
    TextPipeline,
    Token,
    WordTokenizer,
    ENGLISH_STOPWORDS,
    Preprocessor,
    Postprocessor,
    Tokenizer,
)

In [ ]:
dataset = list()

with open('../data/yelp_labelled.txt',
          mode='r',
          encoding='utf-8') as f:
    dataset = f.readlines()

reviews = [re.sub(r'\t[01]\n$', '', review) for review in dataset]

## Part 1: Learn word Tokens one document at a time

Add a code cell immediately below this one. Create a `TextPipeline` instance named `basic_pipeline` with:
- Preprocessor: `CasingNormalizer`, `SpecialCharacterRemover`
- Tokenizer: `WordTokenizer`
- Postprocessor: `StopwordRemover`

Create a new list named `document_tokens` and populate it with the output of `basic_pipeline.process(reveiw, update_vocab=True)` for the first 20 items in `reviews`.

In [ ]:
assert basic_pipeline is not None
assert isinstance(basic_pipeline, TextPipeline)

assert len(document_tokens) == 20

for stopword in ENGLISH_STOPWORDS:
    assert all(Token(stopword) not in document for document in document_tokens)
    assert stopword not in basic_pipeline.vocabulary

assert all(
    token in basic_pipeline.vocabulary for document in document_tokens for token in document)
assert repr(basic_pipeline) == (
    "textpipeline: {steps: {pre: (case_normalizer, special_character_remover), "
    "tokenizer: (word_tokenizer), post: (stopword_remover)}, "
    f"vocab_size: {len(basic_pipeline.vocabulary)}" + "}"
)

## Part 2: Stem and create n-grams

Add a code cell immediately below this one. Create a `TextPipeline` instance named `ngram_pipeline` with:
- Preprocessor: `CasingNormalizer`, `SpecialCharacterRemover`
- Tokenizer: `WordTokenizer`
- Postprocessor: `SimpleStemmer`, `NGramGenerator(2)`

Run it once with `ngram_pipeline.process("I am liking this place.", update_vocab=True)`; store the result in `bigrams`.

using the same preprocessing and tokenizer, then use `SimpleStemmer()` and `NGramGenerator(2)` as postprocessors. Process `"Very good food!"` with `update_vocab=True` and save the result as `bigrams`.

In [ ]:
assert ngram_pipeline is not None
assert isinstance(ngram_pipeline, TextPipeline)

assert bigrams == [Token("i<sep>am"), Token("am<sep>lik"), Token(
    "lik<sep>thi"), Token("thi<sep>place")]

assert set(bigrams) <= ngram_pipeline.vocabulary
assert all("<sep>" in token.value for token in bigrams)
assert repr(ngram_pipeline) == (
    "textpipeline: {steps: {pre: (case_normalizer, special_character_remover), "
    "tokenizer: (word_tokenizer), post: (simple_stemmer, ngram_generator(n=2))}, "
    "vocab_size: 4}"
)

## Part 3: POS-configured lemmatization and frozen vocabulary

Add a code cell immediately below this one. Create a a `TextPipeline` instance named `lemma_pipeline` with:
- Preprocessor: `CasingNormalizer`
- Tokenizer: `WordTokenizer`
- Postprocessor: `Lemmatizer(pos="v")`

Then, using the same pipeline instance, run it two times:
- Once as `lemma_pipeline.process("Running", update_vocab=True)`; store the result in `running_tokens`
- Once as `lemma_pipeline.process("Swimming", update_vocab=False)`; store the result in `swimming_tokens`

NOTE: Don't forget to call `ensure_wordnet()` on your lemmatizer before running your pipeline.

In [ ]:
assert lemma_pipeline is not None
assert isinstance(lemma_pipeline, TextPipeline)
assert running_tokens is not None
assert swimming_tokens is not None

assert Token("run") in lemma_pipeline.vocabulary
assert running_tokens == [Token("run")]
assert Token("swim") not in lemma_pipeline.vocabulary
assert swimming_tokens == [Token("<unk>")]

assert repr(lemma_pipeline) == (
    "textpipeline: {steps: {pre: (case_normalizer), tokenizer: (word_tokenizer), "
    "post: (lemmatizer(pos=v))}, vocab_size: 1}"
)

## Part 4: Add a contraction-expander plugin

One of the benefits of the way `TextPipeline` was designed is that it allows the creation of arbitrary preprocessor, tokenizers, and postprocessors. For example, the following preprocessor can be added to any pipeline to extract the review text from the unmodified `dataset`.

In [ ]:
class ReviewExtractor(Preprocessor):
    def __call__(self, document: str) -> str:
        return re.sub(r'\t[01]\n$', '', document)

    def display_name(self) -> str:
        return "review_extractor"


review_extractor = ReviewExtractor()

display(review_extractor(dataset[0]))

It can also be used in a pipeline:

In [ ]:
sample_pipeline = TextPipeline(
    preprocessors=[ReviewExtractor()],
    tokenizer=WordTokenizer(),
)

display(sample_pipeline.process(dataset[0], update_vocab=True))

In [ ]:
common_contractions = {
    "i'm": "i am",
    "can't": "cannot",
    "don't": "do not",
    "it's": "it is",
    "we're": "we are",
}

Add a code cell immediately below this one. Define `ContractionExpander` by extending `Preprocessor`. Give it a contraction mapping in its constructor and implement both `__call__(self, text: str) -> str` and `display_name(self) -> str`, returning `"contraction_expander"`. Then create `contraction_pipeline` with 
- Preprocessor: `CasingNormalizer`, `ContractionExpander`, `SpecialCharacterRemover`
- Tokenizer: `WordTokenizer`

Then run `contraction_pipeline.process("I'm not happy.", update_vocab=True)`; store the result in `expanded_tokens`.

Use `common_contractions` from the previous cell for a list of contractions and their replacements in your implementation

In [ ]:
assert expanded_tokens == [Token("i"), Token(
    "am"), Token("not"), Token("happy")]
assert set(expanded_tokens) <= contraction_pipeline.vocabulary
assert repr(contraction_pipeline) == (
    "textpipeline: {steps: {pre: (case_normalizer, contraction_expander, "
    "special_character_remover), tokenizer: (word_tokenizer), post: ()}, "
    "vocab_size: 4}"
)

## Reflection

Add a markdown cell immediately below this one, answer the following questions:

1) Why do you think it's important to have the `update_vocab` argument? In which scenario do you imagine it would be valuable to omit unkown tokens from the vocabulary?

2) Why do you think there is a distinction between preprocessing and postprocessing steps?

3) Why do you think n-grams are useful? When are they not?

Feel free to add additional code cells to try out different pipelines